# Creating a Full Sim Visualization

Prompted by desire to have an image in the 

In [1]:
import merlin_spectra
from merlin_spectra.emission import EmissionLineInterpolator
import merlin_spectra.linelists as linedata

import os
import copy

import numpy as np
from numpy import ndarray
import scipy.integrate as spyint
import yt
from yt.frontends.ramses.field_handlers import RTFieldFileHandler

import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
from astropy.constants import c

# Path to the installed package
merlin_path = os.path.dirname(merlin_spectra.__file__)

# Path to the linelists folder inside MERLIN
line_list = os.path.join(merlin_path, "linelists/linelist.dat")

print("MERLIN path:", merlin_path)
print("Line list path:", line_list)

filename = "/Users/lamoreau/python/ASpec/SimulationFiles/output_00273/info_00273.txt"
# need to make this a relative path that is consistant
# should be able to select files with GUI if we are going to have one

lines=["H1_6562.80A","O1_1304.86A","O1_6300.30A","O2_3728.80A","O2_3726.10A",
       "O3_1660.81A","O3_1666.15A","O3_4363.21A","O3_4958.91A","O3_5006.84A", 
       "He2_1640.41A","C2_1335.66A","C3_1906.68A","C3_1908.73A","C4_1549.00A",
       "Mg2_2795.53A","Mg2_2802.71A","Ne3_3868.76A","Ne3_3967.47A",
       "N5_1238.82A",
       "N5_1242.80A","N4_1486.50A","N3_1749.67A","S2_6716.44A","S2_6730.82A"]

wavelengths=np.array([6562.80, 1304.86, 6300.30, 3728.80, 3726.10, 1660.81, 1666.15,
             4363.21, 4958.91, 5006.84, 1640.41, 1335.66,
             1906.68, 1908.73, 1549.00, 2795.53, 2802.71, 3868.76,
             3967.47, 1238.82, 1242.80, 1486.50, 1749.67, 6716.44, 6730.82])

cell_fields = [
    "Density",
    "x-velocity",
    "y-velocity",
    "z-velocity",
    "Pressure",
    "Metallicity",
    "xHI",
    "xHII",
    "xHeII",
    "xHeIII",
]

epf = [
    ("particle_family", "b"),
    ("particle_tag", "b"),
    ("particle_birth_epoch", "d"),
    ("particle_metallicity", "d"),
]

# Ionization Parameter Field
# Based on photon densities in bins 2-4
# Don't include bin 1 -> Lyman Werner non-ionizing
def _ion_param(field, data):
    p = RTFieldFileHandler.get_rt_parameters(ds).copy()
    p.update(ds.parameters)

    cgs_c = 2.99792458e10     #light velocity

    # Convert to physical photon number density in cm^-3
    pd_2 = data['ramses-rt','Photon_density_2']*p["unit_pf"]/cgs_c
    pd_3 = data['ramses-rt','Photon_density_3']*p["unit_pf"]/cgs_c
    pd_4 = data['ramses-rt','Photon_density_4']*p["unit_pf"]/cgs_c

    photon = pd_2 + pd_3 + pd_4

    return photon/data['gas', 'number_density']


def _my_temperature(field, data):
    #y(i): abundance per hydrogen atom
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90
    kB_RAMSES=yt.YTArray(1.3806200e-16,"erg/K") #defined by RAMSES in cooling_module.f90

    dn=data["ramses","Density"].in_cgs()
    pr=data["ramses","Pressure"].in_cgs()
    yHI=data["ramses","xHI"]
    yHII=data["ramses","xHII"]
    yHe = YHE_RAMSES*0.25/XH_RAMSES
    yHeII=data["ramses","xHeII"]*yHe
    yHeIII=data["ramses","xHeIII"]*yHe
    yH2=1.-yHI-yHII
    yel=yHII+yHeII+2*yHeIII
    mu=(yHI+yHII+2.*yH2 + 4.*yHe) / (yHI+yHII+yH2 + yHe + yel)
    return pr/dn * mu * mH_RAMSES / kB_RAMSES


# TODO see if it works in emission.py
# Luminosity field
# Cloudy Intensity obtained assuming height = 1cm
# Return intensity values erg/s/cm**2
# Multiply intensity at each pixel by volume of pixel -> luminosity
def get_luminosity(line):
   def _luminosity(field, data):
      return data['gas', 'flux_' + line]*data['gas', 'volume']
   return copy.deepcopy(_luminosity)


#number density of hydrogen atoms
def _my_H_nuclei_density(field, data):
    dn=data["ramses","Density"].in_cgs()
    XH_RAMSES=0.76 #defined by RAMSES in cooling_module.f90
    YHE_RAMSES=0.24 #defined by RAMSES in cooling_module.f90
    mH_RAMSES=yt.YTArray(1.6600000e-24,"g") #defined by RAMSES in cooling_module.f90

    return dn*XH_RAMSES/mH_RAMSES


def _pressure(field, data):
    if 'hydro_thermal_pressure' in dir(ds.fields.ramses): # and 
        #'Pressure' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_thermal_pressure']


def _xHI(field, data):
    if 'hydro_xHI' in dir(ds.fields.ramses): # and \
        #'xHI' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHI']


def _xHII(field, data):
    if 'hydro_xHII' in dir(ds.fields.ramses): # and \
        #'xHII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHII']


def _xHeII(field, data):
    if 'hydro_xHeII' in dir(ds.fields.ramses): # and \
        #'xHeII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeII']


def _xHeIII(field, data):
    if 'hydro_xHeIII' in dir(ds.fields.ramses): # and \
        #'xHeIII' not in dir(ds.fields.ramses):
        return data['ramses', 'hydro_xHeIII']

'''
-------------------------------------------------------------------------------
Load Simulation Data
Add Derived Fields
-------------------------------------------------------------------------------
'''

ds = yt.load(filename, extra_particle_fields=epf)

ds.add_field(
    ("gas","number_density"),
    function=_my_H_nuclei_density,
    sampling_type="cell",
    units="1/cm**3",
    force_override=True
)


ds.add_field(
    ("ramses","Pressure"),
    function=_pressure,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHI"),
    function=_xHI,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHII"),
    function=_xHII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHeII"),
    function=_xHeII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("ramses","xHeIII"),
    function=_xHeIII,
    sampling_type="cell",
    units="1",
    #force_override=True
)

ds.add_field(
    ("gas","my_temperature"),
    function=_my_temperature,
    sampling_type="cell",
    # TODO units
    #units="K",
    #units="K*cm**3/erg",
    units='K*cm*dyn/erg',
    force_override=True
)

# Ionization parameter
ds.add_field(
    ('gas', 'ion_param'),
    function=_ion_param,
    sampling_type="cell",
    units="cm**3",
    force_override=True
)

ds.add_field(
    ("gas","my_H_nuclei_density"),
    function=_my_H_nuclei_density,
    sampling_type="cell",
    units="1/cm**3",
    force_override=True
)

# Normalize by Density Squared Flag
dens_normalized = True
if dens_normalized: 
    units = '1/cm**6'
else:
    units = '1'


print(lines)
print(len(lines))

# Instance of EmissionLineInterpolator for line list at filename
# print(line_list) #see cell 2 above for details
emission_interpolator = EmissionLineInterpolator(lines, line_list) #why is this interpolated? computational speedup?

# Add flux and luminosity fields for all lines in the list
for i, line in enumerate(lines):
    ds.add_field(
        ('gas', 'flux_' + line),
        function=emission_interpolator.get_line_emission(
            i, dens_normalized=dens_normalized
        ),
        sampling_type='cell',
        units=units,
        force_override=True
    )
    # TODO change get_line_emission to accept line not idx

    ds.add_field(
        ('gas', 'luminosity_' + line),
        function=emission_interpolator.get_luminosity(lines[i]),
        #function=get_luminosity(lines[i]),
        sampling_type='cell',
        units='1/cm**3',
        force_override=True
    )

zsource = ds.current_redshift

# From sedcontinuum file, this is what I will assume 
# is the cosmology used to create the simulation
cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Tcmb0=2.725 * u.K, Om0=0.3)

# --------------------------------------------
# INPUT: list of emission lines (exact names)
# --------------------------------------------
lines = np.array([
    'H1 6562.80A', 'O1 1304.86A', 'O1 6300.30A', 'O2 3728.80A',
    'O2 3726.10A', 'O3 1660.81A', 'O3 1666.15A', 'O3 4363.21A',
    'O3 4958.91A', 'O3 5006.84A', 'He2 1640.41A', 'C2 1335.66A',
    'C3 1906.68A', 'C3 1908.73A', 'C4 1549.00A', 'Mg2 2795.53A',
    'Mg2 2802.71A', 'Ne3 3868.76A', 'Ne3 3967.47A', 'N5 1238.82A',
    'N5 1242.80A', 'N4 1486.50A', 'N3 1749.67A', 'S2 6716.44A',
    'S2 6730.82A'
])
# --------------------------------------------
# Convert "O1 1304.86A" → "O1_1304.86A"
# --------------------------------------------
lines = np.array([l.replace(" ", "_") for l in lines])

# -------------------------------------------------------------------------
# PREP: load all gas data and cell volume once (faster than inside loop)
# -------------------------------------------------------------------------
ad = ds.all_data()
# ----------------------------------------------------
# OUTPUT ARRAYS
# ----------------------------------------------------
total_fluxes = []
total_luminosities = []

# creating proj of a boxed region
cx = np.mean(ad["star", "particle_position_x"])
cy = np.mean(ad["star", "particle_position_y"])
cz = np.mean(ad["star", "particle_position_z"])
center = [cx, cy, cz]
halfa = ds.quan(20, "kpc")

low_edge = [center[0] - halfa, center[1] - halfa, center[2] - halfa]
high_edge = [center[0] + halfa, center[1] + halfa, center[2] + halfa]

cube_region = ds.region(center, low_edge, high_edge)

ad_box = cube_region


cell_volume_box = ad_box["cell_volume"]   # in cm^3
# ----------------------------------------------------
# OUTPUT ARRAYS
# ----------------------------------------------------
total_fluxes_box = []
total_luminosities_box = []


cell_vol  = cell_volume_box.to("cm**3").d[None, :]     # broadcast to shape (1, Ncells)

MERLIN path: /Users/lamoreau/pyS99/pyS99/.venv/lib/python3.10/site-packages/merlin_spectra
Line list path: /Users/lamoreau/pyS99/pyS99/.venv/lib/python3.10/site-packages/merlin_spectra/linelists/linelist.dat


yt : [INFO     ] 2026-04-06 10:56:19,808 Parameters: current_time              = 0.3604448649237178 Gyr
yt : [INFO     ] 2026-04-06 10:56:19,809 Parameters: domain_dimensions         = [64 64 64]
yt : [INFO     ] 2026-04-06 10:56:19,810 Parameters: domain_left_edge          = [0. 0. 0.]
yt : [INFO     ] 2026-04-06 10:56:19,812 Parameters: domain_right_edge         = [1. 1. 1.]
yt : [INFO     ] 2026-04-06 10:56:19,813 Parameters: cosmological_simulation   = True
yt : [INFO     ] 2026-04-06 10:56:19,815 Parameters: current_redshift          = 12.171087046255657
yt : [INFO     ] 2026-04-06 10:56:19,816 Parameters: omega_lambda              = 0.685000002384186
yt : [INFO     ] 2026-04-06 10:56:19,817 Parameters: omega_matter              = 0.314999997615814
yt : [INFO     ] 2026-04-06 10:56:19,818 Parameters: omega_radiation           = 0.0
yt : [INFO     ] 2026-04-06 10:56:19,819 Parameters: hubble_constant           = 0.674000015258789
yt : [WARNING  ] 2026-04-06 10:56:19,914 This output

['H1_6562.80A', 'O1_1304.86A', 'O1_6300.30A', 'O2_3728.80A', 'O2_3726.10A', 'O3_1660.81A', 'O3_1666.15A', 'O3_4363.21A', 'O3_4958.91A', 'O3_5006.84A', 'He2_1640.41A', 'C2_1335.66A', 'C3_1906.68A', 'C3_1908.73A', 'C4_1549.00A', 'Mg2_2795.53A', 'Mg2_2802.71A', 'Ne3_3868.76A', 'Ne3_3967.47A', 'N5_1238.82A', 'N5_1242.80A', 'N4_1486.50A', 'N3_1749.67A', 'S2_6716.44A', 'S2_6730.82A']
25
/Users/lamoreau/pyS99/pyS99/.venv/lib/python3.10/site-packages/merlin_spectra/linelists/linelist-all.dat
/Users/lamoreau/pyS99/pyS99/.venv/lib/python3.10/site-packages/merlin_spectra/linelists/linelist-all.dat
minU=-9.0, maxU=2.0, stepU=0.5, minN=-4.0, maxN=7.0, stepN=0.5, minT=1.0, maxT=8.0, stepT=0.2
Line List Shape = (26, 19044)
23 23 36


yt : [INFO     ] 2026-04-06 10:56:21,916 Identified   384/  384 intersecting domains (  385 through hilbert key indexing)
yt : [INFO     ] 2026-04-06 10:56:23,293 Identified   384/  384 intersecting domains (  384 through hilbert key indexing)


In [2]:
print(ds.field_list)

[('gravity', 'Potential'), ('gravity', 'x-acceleration'), ('gravity', 'y-acceleration'), ('gravity', 'z-acceleration'), ('io', 'particle_birth_epoch'), ('io', 'particle_family'), ('io', 'particle_identity'), ('io', 'particle_mass'), ('io', 'particle_metallicity'), ('io', 'particle_position_x'), ('io', 'particle_position_y'), ('io', 'particle_position_z'), ('io', 'particle_refinement_level'), ('io', 'particle_tag'), ('io', 'particle_velocity_x'), ('io', 'particle_velocity_y'), ('io', 'particle_velocity_z'), ('nbody', 'particle_mass'), ('nbody', 'particle_position_x'), ('nbody', 'particle_position_y'), ('nbody', 'particle_position_z'), ('nbody', 'particle_velocity_x'), ('nbody', 'particle_velocity_y'), ('nbody', 'particle_velocity_z'), ('ramses', 'Density'), ('ramses', 'Metallicity'), ('ramses', 'Pressure'), ('ramses', 'hydro_i_level'), ('ramses', 'hydro_xHI'), ('ramses', 'hydro_xHII'), ('ramses', 'hydro_xHeII'), ('ramses', 'hydro_xHeIII'), ('ramses', 'x-velocity'), ('ramses', 'y-velocit

In [3]:
print(('ramses', 'Density') in ds.field_list)

ad = ds.all_data()
print(ad[('ramses','Density')].min(), ad[('ramses','Density')].max())

True


yt : [INFO     ] 2026-04-06 10:56:31,800 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)


0.02478099477546934 code_density 44854766.03650063 code_density


In [ ]:
sc = yt.create_scene(ds, lens_type = "perspective")
source = sc[0]

source.set_field(('gas', 'density')) #set the field to render
source.set_log(True) #use log space scaling

bounds = (3e-28, 5e-19) #no idea if this is right, may need to adjust
tf = yt.ColorTransferFunction(x_bounds=np.log10(bounds))
help(yt.ColorTransferFunction)

tf.add_gaussian(location = -27, width = 0.003, height = [1,0,0,1])
tf.add_gaussian(location = -26, width = 0.003, height = [0.5,1,1,0.5])
print(tf)
source.tfh.tf = tf #tfh = TransferFunctionHelper
source.tfh.bounds = bounds
source.tfh.plot('transferFunction.png', profile_field=('gas', 'density'))

sc.save('volume.png', sigma_clip = 4)


yt : [INFO     ] 2026-04-06 15:28:47,414 Setting default field to ('gas', 'density')


Help on class ColorTransferFunction in module yt.visualization.volume_rendering.transfer_functions:

class ColorTransferFunction(MultiVariateTransferFunction)
 |  ColorTransferFunction(x_bounds, nbins=256, grey_opacity=False)
 |  
 |  A complete set of transfer functions for standard color-mapping.
 |  
 |  This is the best and easiest way to set up volume rendering.  It
 |  creates field tables for all three colors, their alphas, and has
 |  support for sampling color maps and adding independent color values at
 |  all locations.  It will correctly set up the
 |  `MultiVariateTransferFunction`.
 |  
 |  Parameters
 |  ----------
 |  x_bounds : tuple of floats
 |      The min and max for the transfer function.  Values below or above
 |      these values are discarded.
 |  nbins : int
 |      How many bins to calculate; in between, linear interpolation is
 |      used, so low values are typically fine.
 |  grey_opacity : bool
 |      Should opacity be calculated on a channel-by-channel 

yt : [INFO     ] 2026-04-06 15:28:48,093 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)
yt : [WARNING  ] 2026-04-06 15:28:50,504 No previously rendered image found, rendering now.
yt : [INFO     ] 2026-04-06 15:28:50,504 Rendering scene (Can take a while).
yt : [INFO     ] 2026-04-06 15:28:50,505 Creating volume
yt : [INFO     ] 2026-04-06 15:28:51,047 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)
yt : [INFO     ] 2026-04-06 15:28:55,351 Identified 1 intersecting domains
yt : [WARNING  ] 2026-04-06 15:29:00,739 RAMSESDomainSubset (info_00273): , base_region=YTRegion (info_00273): , center=[5.98491924e+24 5.98491924e+24 5.98491924e+24] cm, left_edge=[0. 0. 0.] cm, right_edge=[1.19698385e+25 1.19698385e+25 1.19698385e+25] cm, domain=RAMSESDomainFile: 1, ds=info_00273.retrieve_ghost_zones was called with the `smoothed` argument set to True. This is not supported, ignoring it.
yt : [INFO     ] 2026-04-06 15:29:00,743 Iden

In [ ]:
sc = yt.create_scene(ds, lens_type = "perspective")
source = sc[0]

source.set_field(('gas', 'density')) #set the field to render
source.set_log(True) #use log space scaling

bounds = (3e-25, 5e-19)

tf = yt.ColorTransferFunction(x_bounds=np.log10(bounds))
tf.clear()

# evenly spaced in log space
tf.add_layers(8, colormap='inferno')

source.tfh.tf = tf #tfh = TransferFunctionHelper
source.tfh.bounds = bounds
source.tfh.plot('transferFunction.png', profile_field=('gas', 'density'))

sc.save('volume.png', sigma_clip = 4)

yt : [INFO     ] 2026-04-06 15:40:04,744 Setting default field to ('gas', 'density')
yt : [INFO     ] 2026-04-06 15:40:07,798 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)
yt : [WARNING  ] 2026-04-06 15:40:10,905 No previously rendered image found, rendering now.
yt : [INFO     ] 2026-04-06 15:40:10,906 Rendering scene (Can take a while).
yt : [INFO     ] 2026-04-06 15:40:10,908 Creating volume
yt : [INFO     ] 2026-04-06 15:40:11,481 Identified   162/  384 intersecting domains (  385 through hilbert key indexing)
yt : [INFO     ] 2026-04-06 15:40:15,650 Identified 1 intersecting domains
yt : [WARNING  ] 2026-04-06 15:40:21,240 RAMSESDomainSubset (info_00273): , base_region=YTRegion (info_00273): , center=[5.98491924e+24 5.98491924e+24 5.98491924e+24] cm, left_edge=[0. 0. 0.] cm, right_edge=[1.19698385e+25 1.19698385e+25 1.19698385e+25] cm, domain=RAMSESDomainFile: 1, ds=info_00273.retrieve_ghost_zones was called with the `smoothed` argument set to Tr

In [28]:
cam = sc.camera
cam.focus = ds.domain_center
#cam.position = ds.domain_center + 3.0 * ds.domain_width
cam.width = 0.1 * ds.domain_width
cam.resolution = (800, 800)

sc.save("debug.png", sigma_clip=4.0)

yt : [WARNING  ] 2026-04-06 15:33:24,861 Previously rendered image exists, but rendering anyway. Supply 'render=False' to save previously rendered image directly.
yt : [INFO     ] 2026-04-06 15:33:24,864 Rendering scene (Can take a while).
yt : [INFO     ] 2026-04-06 15:34:03,605 Saving rendered image to debug.png
